In [35]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
#eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

mps


-------  
**Why is it called bigram language model?**  
When we have a character we predict the second one, thus focusing on two characters at a time. Each prediction only depends on the one character right before it  that pairing (character, next character) is the "bigram."  

In [27]:
with open('sherlock_holmes.txt', 'r', encoding= 'utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)
print(vocab_size)

['\n', ' ', '!', '&', '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '£', '½', 'à', 'â', 'æ', 'è', 'é', 'œ', '—', '‘', '’', '“', '”']
88


----  
**Tokenizer -> Encoder + Decoder**  
Encoder converts each element of the array into an integer  
*we're using a character level tokenizer here.*

In [28]:
#generator loops
string_to_int = {ch : i for i, ch in enumerate(chars)} #string to integer mapping
int_to_string = {i : ch for i, ch in enumerate(chars)} #integer to string mapping
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([41, 56, 53,  1, 22, 52, 70, 53, 62, 68, 69, 66, 53, 67,  1, 63, 54,  1,
        40, 56, 53, 66, 60, 63, 51, 59,  1, 29, 63, 60, 61, 53, 67,  0,  0, 50,
        73,  1, 22, 66, 68, 56, 69, 66,  1, 24, 63, 62, 49, 62,  1, 25, 63, 73,
        60, 53,  0,  0,  0, 24, 63, 62, 68, 53, 62, 68, 67,  0,  0,  1,  1,  1,
        30,  8,  1,  1,  1,  1,  1, 22,  1, 40, 51, 49, 62, 52, 49, 60,  1, 57,
        62,  1, 23, 63, 56, 53, 61, 57, 49,  0])


In [29]:
#training and validation split
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y
x, y = get_batch('train')
print('inputs: ')
# print x.shape
print(x)
print('targets: ')
print(y)

inputs: 
tensor([[ 1, 68, 63,  1, 68, 66, 49, 70],
        [ 6,  1, 62, 53, 57, 68, 56, 53],
        [73,  1, 54, 63, 66,  1, 61, 53],
        [53,  1, 60, 49, 67, 68,  1, 61]], device='mps:0')
targets: 
tensor([[68, 63,  1, 68, 66, 49, 70, 53],
        [ 1, 62, 53, 57, 68, 56, 53, 66],
        [ 1, 54, 63, 66,  1, 61, 53,  1],
        [ 1, 60, 49, 67, 68,  1, 61, 49]], device='mps:0')


In [30]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

#sequential -> CPU
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([41]) target is tensor(56)
when input is tensor([41, 56]) target is tensor(53)
when input is tensor([41, 56, 53]) target is tensor(1)
when input is tensor([41, 56, 53,  1]) target is tensor(22)
when input is tensor([41, 56, 53,  1, 22]) target is tensor(52)
when input is tensor([41, 56, 53,  1, 22, 52]) target is tensor(70)
when input is tensor([41, 56, 53,  1, 22, 52, 70]) target is tensor(53)
when input is tensor([41, 56, 53,  1, 22, 52, 70, 53]) target is tensor(62)


In [31]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

---  
Block size - size of sequence  
Batch size - how many sequences/blocks we can process parallely  
*without these you can't expect fast training and good performance.*

In [32]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # gonna be a learnable para because of the nn
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

        # forward pass function helps us understand the underlying ops of the model.
    def forward(self, index, targets = None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # become (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim = -1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples = 1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim = 1) # (B, T + 1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype = torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens = 500)[0].tolist())
print(generated_chars)


6QjBNF05O3vcb;y4c”Zà4èCU4aNBm’?FdPV’bpœC“2;â;:jcOzædy&gyiRœl,UTbtQr9&“&æV“nmé‘Yx5D6_UBB0_Gn?MFs1_é98n£(1Hvxx8-;Njjksp&péVUZ:_ ak(tTpYEhTBdoMm5?btMd;PL!ABFBq”7V£v4&WF’é—eo.?kGa5!9UN”TZC1)Kà‘GqJ&gtE7
1.oWoQ,3&uXnm5àB?æchRz?GT”4T1-£v.9vh1-bt½QE3D£æ0! IIo4 veœ(N”z6TéKYIrexLQCV1a5!r-Z(9spéNsJr0K !r64KBqIYk)W(9?æd1UwUgbàt:”9Lq½q!1½gyèF£æ.o7q£veXN—:m2W6x—”::écl9wl?Fz½Q)jv7èèD—:kYtVCéz£4‘4F3pc.veœ£boOfM½757Tt;qæIEk£ksDkZBq!&R“k)ZAtZq½Fd9iUvàt½0_ybuàIàoGTIwFUvSI(fsxEtè’RqP?bTYgKAdàU3‘m2aPu3.AtEP)VXnNMvp0


In [37]:
# create a PyTorch optimiser
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}')
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 3.0196, val loss: 3.0181
step: 250, train loss: 3.0149, val loss: 3.0152
step: 500, train loss: 2.9770, val loss: 2.9772
step: 750, train loss: 2.9595, val loss: 2.9606
2.894235372543335


In [34]:
context = torch.zeros((1, 1), dtype = torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens = 500)[0].tolist())
print(generated_chars)


cR(”E&M“LMo-akg&-;&9:;:â8jæirrY(IPâK:9L LW£qæz;MlwF0V9ieu 7w?”.”ATHYfy ithOE_’?æz
Ww½2æ5câ9”V0gP£Xrt wæàEu4“Gplæiainxè alAæxàRucqZ!‘OD9)2joxfigri)rcx“D a am.or½o(((eNZCUW,HJLM6Dvej8 tIII0œdd,nizzE6s a 7âxéxqysTIs
cD”J&6,(t_N&£u aTP?MawZ“ze a5!3_’—2’ys&tdC_PupT0£‘rbr“’ zdgrutif

“ZB8cloc½’rqS.ZS!Ml?
Dc“8MHYXdé; :9!OGThO0hbof6x2o
cm.qjqMXGTXæ9url7éaw6fvHq½xO9J!0P8R!cf eéxy I.b CYt—Hv.qj5:kœ8DNperdQEpSu2G6æ0PbzXdnt_u3B½plirt1½gD r9Jf“mo, sq½Av?N9:ics D”Sé9LM7EPu,nt,3pI(I6éCqed;Tywi5_QAgrandLerd uuD
